# 2) 2-Minute Rescue Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.

## G — Goals

When a student is stuck, select one 2-minute rescue script and track which scripts helped.

## Simple rules (policy)

- Pick one rescue script each time.
- Ask “Did it help? yes/no”.
- Track helped/total stats per script in memory.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).

In [ ]:
from __future__ import annotations

import json, os, random, time, csv
from dataclasses import dataclass
from typing import Any, Dict, List
from datetime import datetime

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    tool_name: str
    args: Dict[str, Any]

class Tools:
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        try:
            return input(f"\n[You] {prompt}\n> ").strip()
        except EOFError:
            return ''

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

    @staticmethod
    def save_csv(path: str, stats: Dict[str, Any]) -> str:
        try:
            with open(path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['script', 'helped', 'total', 'success_rate'])
                for name, d in sorted(stats.items(), key=lambda kv: -((int(kv[1].get('helped',0))/max(1,int(kv[1].get('total',0)))) if int(kv[1].get('total',0))>0 else 0)):
                    helped = int(d.get('helped', 0))
                    total = int(d.get('total', 0))
                    rate = helped / total if total > 0 else 0.0
                    writer.writerow([name, helped, total, f"{rate:.2f}"])
            return 'csv_saved'
        except Exception as e:
            print('Error saving CSV:', e)
            return 'csv_error'

class Environment:
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == 'notify':
            Tools.notify(str(action.args.get('message', '')))
            return {'result': 'notified', 'memory': memory}

        if action.tool_name == 'ask':
            ans = Tools.ask(str(action.args.get('prompt', '')))
            return {'result': ans, 'memory': memory}

        if action.tool_name == 'load_memory':
            loaded = Tools.load_memory(self.memory_path)
            return {'result': 'loaded', 'memory': loaded}

        if action.tool_name == 'save_memory':
            Tools.save_memory(self.memory_path, memory)
            return {'result': 'saved', 'memory': memory}

        if action.tool_name == 'export_csv':
            stats = memory.get('stats', {})
            csv_path = os.path.splitext(self.memory_path)[0] + '.csv'
            res = Tools.save_csv(csv_path, stats)
            return {'result': ('exported_csv' if res == 'csv_saved' else 'csv_error'), 'memory': memory}

        if action.tool_name == 'terminate':
            return {'result': 'terminated', 'memory': memory}

        return {'result': 'unknown_tool', 'memory': memory}


In [ ]:
# Implementation (GAME Agent Loop)
MEMORY_FILE = '02_rescue_memory.json'

# Demo mode shortens the 2-minute timer for classroom demos
DEMO_MODE = True
COUNTDOWN_SECS = 5 if DEMO_MODE else 120

SCRIPTS = [
    ('Rubber Duck', [
        'Explain the problem out loud in 3 sentences.',
        'Name the exact input and expected output.',
        'Identify the first point you feel uncertain.',
    ]),
    ('Minimal Example', [
        'Create the smallest input that triggers the bug.',
        'Remove everything not needed to reproduce it.',
        'Test again with only the minimal case.',
    ]),
    ('First Error First', [
        'Find the first incorrect value in your trace/output.',
        'Ask: what line produced it?',
        'Check assumptions right before that line.',
    ]),
    ('Base Case Check', [
        'If recursion: write the base case in plain English.',
        'Test the base case with 1 simple input.',
        'Then test 1 step above base case.',
    ]),
    ('Print & Probe', [
        'Add ONE print/log at the key decision point.',
        'Run once and observe the value.',
        'Remove the print after learning.',
    ]),
]

def pick_script(memory: Dict[str, Any]) -> int:
    stats = memory.get('stats', {})
    names = [s[0] for s in SCRIPTS]
    favorites = set(memory.get('favorites', []))
    weights = []
    for n in names:
        d = stats.get(n, {'helped': 0, 'total': 0})
        helped, total = int(d.get('helped', 0)), int(d.get('total', 0))
        base = (helped + 1) / (total + 2)  # smoothed
        boost = 1.4 if n in favorites else 1.0
        weights.append(base * boost)
    return random.choices(range(len(SCRIPTS)), weights=weights, k=1)[0]

def _format_script(name: str, steps: List[str]) -> str:
    return f"{name}:\n" + '\n'.join([f"- {s}" for s in steps])

def decide_next_action(user_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(user_text):
        return Action('terminate', {})

    cmd = (user_text or '').strip()
    cmd_lower = cmd.lower()

    # add: Name | step1 ; step2 ; step3
    if cmd_lower.startswith('add:'):
        payload = cmd.split(':', 1)[1].strip()
        parts = [p.strip() for p in payload.split('|', 1)]
        name = parts[0] if parts else 'New Script'
        steps = []
        if len(parts) > 1:
            steps = [s.strip() for s in parts[1].split(';') if s.strip()]
        SCRIPTS.append((name, steps))
        return Action('notify', {'message': f'Added script: {name} ({len(steps)} steps)'})

    # remove: INDEX or remove: NAME
    if cmd_lower.startswith('remove:'):
        payload = cmd.split(':', 1)[1].strip()
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(SCRIPTS):
                removed = SCRIPTS.pop(idx)
                favs = set(memory.get('favorites', []))
                if removed[0] in favs:
                    favs.discard(removed[0])
                    memory['favorites'] = list(favs)
                return Action('notify', {'message': f'Removed: {removed[0]}'})
            return Action('notify', {'message': 'Index out of range.'})
        for i, s in enumerate(SCRIPTS):
            if s[0].lower() == payload.lower():
                removed = SCRIPTS.pop(i)
                favs = set(memory.get('favorites', []))
                if removed[0] in favs:
                    favs.discard(removed[0])
                    memory['favorites'] = list(favs)
                return Action('notify', {'message': f'Removed: {removed[0]}'})
        return Action('notify', {'message': 'Script not found.'})

    if cmd_lower in {'list', 'scripts'}:
        lines = [f"{i+1}. {s[0]} ({len(s[1])} steps)" for i, s in enumerate(SCRIPTS)]
        return Action('notify', {'message': 'Scripts:\n' + '\n'.join(lines)})

    if cmd_lower in {'best', 'top', 'best_script'}:
        stats = memory.get('stats', {})
        if not stats:
            return Action('notify', {'message': 'No stats yet.'})
        def rate(d):
            t = int(d.get('total', 0))
            return int(d.get('helped', 0)) / t if t > 0 else 0.0
        best = max(stats.items(), key=lambda kv: (rate(kv[1]), int(kv[1].get('total',0))))
        name, d = best
        return Action('notify', {'message': f'Best script: {name} -> helped={d.get('helped',0)} total={d.get('total',0)}'})

    if cmd_lower.startswith('export'):
        return Action('export_csv', {})

    if cmd_lower.startswith('fav:') or cmd_lower.startswith('favorite:'):
        payload = cmd.split(':', 1)[1].strip() if ':' in cmd else ''
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(SCRIPTS):
                name = SCRIPTS[idx][0]
                favs = set(memory.get('favorites', []))
                if name in favs:
                    favs.discard(name)
                    action_msg = f'Unfavorited: {name}'
                else:
                    favs.add(name)
                    action_msg = f'Favorited: {name}'
                memory['favorites'] = list(favs)
                return Action('notify', {'message': action_msg})
        return Action('notify', {'message': 

    if cmd_lower in {'favorites', 'favs'}:
        favs = memory.get('favorites', [])
        if not favs:
            return Action('notify', {'message': 'No favorites set.'})
        return Action('notify', {'message': 'Favorites:\n' + '\n'.join(favs)})

    if cmd_lower.startswith('search:'):
        q = cmd.split(':', 1)[1].strip().lower() if ':' in cmd else ''
        if not q:
            return Action('notify', {'message': 'Usage: search: keyword'})
        matches = [f"{i+1}. {s[0]}" for i, s in enumerate(SCRIPTS) if q in s[0].lower() or any(q in step.lower() for step in s[1])]
        if not matches:
            return Action('notify', {'message': 'No matches.'})
        return Action('notify', {'message': 'Matches:\n' + '\n'.join(matches)})

    if cmd_lower.startswith('show:'):
        payload = cmd.split(':', 1)[1].strip() if ':' in cmd else ''
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(SCRIPTS):
                name, steps = SCRIPTS[idx]
                return Action('notify', {'message': _format_script(name, steps)})
            return Action('notify', {'message': 'Index out of range.'})
        return Action('notify', {'message': 'Usage: show: INDEX'})

    # default: pick a script
    idx = pick_script(memory)
    name, steps = SCRIPTS[idx]
    memory['last_script'] = name
    msg = _format_script(name, steps)
    return Action('notify', {'message': msg})

def _countdown(secs: int):
    try:
        for i in range(secs, 0, -1):
            print(f'  Starting in {i}...', end='
')
            time.sleep(1)
        print(' ' * 40, end='
')
    except KeyboardInterrupt:
        print()

def run_agent():
    env = Environment(MEMORY_FILE)
    memory = env.execute(Action('load_memory', {}), {})['memory'] or {}

    Tools.notify('2-Minute Rescue Agent is running.')
    Tools.notify(
)

    user_text = Tools.ask(
)
    while True:
        action = decide_next_action(user_text, memory)
        out = env.execute(action, memory)
        memory = out['memory']

        if out['result'] == 'terminated':
            env.execute(Action('save_memory', {}), memory)
            Tools.notify('Saved rescue stats. Bye!')
            break

        if out['result'] == 'exported_csv':
            Tools.notify('Exported stats to CSV.')

        if out['result'] == 'notified':
            # show countdown then ask for feedback
            _countdown(COUNTDOWN_SECS)
            ans = Tools.ask('Did it help? (yes/no)')
            stats = memory.get('stats', {})
            name = memory.get('last_script', 'unknown')
            d = stats.get(name, {'helped': 0, 'total': 0})
            d['total'] = int(d.get('total', 0)) + 1
            if ans.lower().startswith('y'):
                d['helped'] = int(d.get('helped', 0)) + 1
            stats[name] = d
            memory['stats'] = stats

        env.execute(Action('save_memory', {}), memory)
        user_text = Tools.ask('Another block? (or stop / commands)')

run_agent()

## Easy extensions

- Add a countdown timer (set `DEMO_MODE=True` to shorten).
- Let students add/remove their own rescue scripts at runtime using `add:` and `remove:`.
- Show best success-rate script via `best` command.
- Export script success stats to CSV with `export`.
- List current scripts and indices using `list` or `scripts`.
- Mark favorites with `fav: INDEX` and list them with `favorites` (favorites are weighted higher when picking).
- Search scripts by keyword using `search: KEYWORD`.
- Show steps for a script using `show: INDEX`.

Usage examples:

- `add: Quick Fix | Try a small change; Run tests` -> adds a script with two steps.
- `remove: 3` -> removes the 3rd script.
- `fav: 2` -> toggle favorite for script #2.
- `favorites` -> show favorited scripts.
- `search: minimal` -> find scripts matching 'minimal'.
- `show: 1` -> display steps for script #1.
- `export` -> writes `02_rescue_memory.csv` with success rates.

# 2) 2-Minute Rescue Agent

This notebook builds a very small, engaging Python "agent" using the GAME structure:

- G — Goals: what success looks like and simple rules
- A — Actions: the tools the agent can call (plain Python functions)
- M — Memory: small persistent state (saved to a JSON file)
- E — Environment: executes tools and returns feedback to the agent loop

Learning objective: Students understand the agent loop (decide -> act -> observe -> repeat) with minimal coding.

Tip for class demos: If the agent uses waiting/timers, enable DEMO_MODE=True to shorten waits.


## G — Goals

When a student is stuck, select one 2-minute rescue script and track which scripts helped.

## Simple rules (policy)

- Pick one rescue script each time.
- Ask “Did it help? yes/no”.
- Track helped/total stats per script in memory.

## How to run

1. Run all cells top-to-bottom.
2. When the agent loop starts, interact in the notebook input prompts.
3. Stop anytime by typing: `stop` (or `exit`, `quit`).


In [1]:
from __future__ import annotations

import json, os, random, time
from dataclasses import dataclass
from typing import Any, Dict, List, Optional

def should_stop(text: str) -> bool:
    return text.strip().lower() in {"stop", "exit", "quit", "end"}

@dataclass
class Action:
    # Structured action output: tool_name + args
    tool_name: str
    args: Dict[str, Any]

class Tools:
    # A: Actions (Tools) - tiny functions the agent can call
    @staticmethod
    def notify(message: str) -> str:
        print(f"\n[Agent] {message}")
        return "ok"

    @staticmethod
    def ask(prompt: str) -> str:
        return input(f"\n[You] {prompt}\n> ").strip()

    @staticmethod
    def load_memory(path: str) -> Dict[str, Any]:
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as f:
                return json.load(f)
        return {}

    @staticmethod
    def save_memory(path: str, data: Dict[str, Any]) -> str:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        return "saved"

    @staticmethod
    def save_csv(path: str, stats: Dict[str, Any]) -> str:
        try:
            import csv
            with open(path, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerow(['script', 'helped', 'total', 'success_rate'])
                for name, d in sorted(stats.items(), key=lambda kv: -((int(kv[1].get('helped',0))/max(1,int(kv[1].get('total',0)))) if int(kv[1].get('total',0))>0 else 0)):
                    helped = int(d.get('helped', 0))
                    total = int(d.get('total', 0))
                    rate = helped / total if total > 0 else 0.0
                    writer.writerow([name, helped, total, f"{rate:.2f}"])
            return 'csv_saved'
        except Exception as e:
            print('Error saving CSV:', e)
            return 'csv_error'

class Environment:
    # E: Environment - runs tools and returns feedback + updated memory
    def __init__(self, memory_path: str):
        self.memory_path = memory_path

    def execute(self, action: Action, memory: Dict[str, Any]) -> Dict[str, Any]:
        if action.tool_name == "notify":
            Tools.notify(str(action.args.get("message", "")))
            return {"result": "notified", "memory": memory}

        if action.tool_name == "ask":
            ans = Tools.ask(str(action.args.get("prompt", "")))
            return {"result": ans, "memory": memory}

        if action.tool_name == "load_memory":
            loaded = Tools.load_memory(self.memory_path)
            return {"result": "loaded", "memory": loaded}

        if action.tool_name == "save_memory":
            Tools.save_memory(self.memory_path, memory)
            return {"result": "saved", "memory": memory}

        if action.tool_name == "export_csv":
            stats = memory.get('stats', {})
            csv_path = os.path.splitext(self.memory_path)[0] + '.csv'
            res = Tools.save_csv(csv_path, stats)
            return {"result": ("exported_csv" if res == 'csv_saved' else 'csv_error'), "memory": memory}

        if action.tool_name == "terminate":
            return {"result": "terminated", "memory": memory}

        return {"result": "unknown_tool", "memory": memory}


## Implementation (GAME Agent Loop)
The code below is intentionally small and readable.

In [2]:
MEMORY_FILE = "02_rescue_memory.json"

# Demo mode shortens the 2-minute timer for classroom demos
DEMO_MODE = True
COUNTDOWN_SECS = 5 if DEMO_MODE else 120

SCRIPTS = [
    ("Rubber Duck", [
        "Explain the problem out loud in 3 sentences.",
        "Name the exact input and expected output.",
        "Identify the first point you feel uncertain.",
    ]),
    ("Minimal Example", [
        "Create the smallest input that triggers the bug.",
        "Remove everything not needed to reproduce it.",
        "Test again with only the minimal case.",
    ]),
    ("First Error First", [
        "Find the first incorrect value in your trace/output.",
        "Ask: what line produced it?",
        "Check assumptions right before that line.",
    ]),
    ("Base Case Check", [
        "If recursion: write the base case in plain English.",
        "Test the base case with 1 simple input.",
        "Then test 1 step above base case.",
    ]),
    ("Print & Probe", [
        "Add ONE print/log at the key decision point.",
        "Run once and observe the value.",
        "Remove the print after learning.",
    ]),
]

def pick_script(memory: Dict[str, Any]) -> int:
    stats = memory.get("stats", {})
    names = [s[0] for s in SCRIPTS]
    favorites = set(memory.get("favorites", []))
    weights = []
    for n in names:
        d = stats.get(n, {"helped": 0, "total": 0})
        helped, total = int(d.get("helped", 0)), int(d.get("total", 0))
        base = (helped + 1) / (total + 2)  # smoothed
        # boost if user favorited this script
        boost = 1.4 if n in favorites else 1.0
        weights.append(base * boost)
    return random.choices(range(len(SCRIPTS)), weights=weights, k=1)[0]

def _format_script(name: str, steps: List[str]) -> str:
    return f"{name}:\n" + "\n".join([f"- {s}" for s in steps])

def decide_next_action(user_text: str, memory: Dict[str, Any]) -> Action:
    if should_stop(user_text):
        return Action("terminate", {})

    cmd = (user_text or "").strip()
    cmd_lower = cmd.lower()
    # command handlers
    if cmd_lower.startswith("add:"):
        # Format: add: Name | step1 ; step2 ; step3
        payload = cmd.split(":", 1)[1].strip()
        parts = [p.strip() for p in payload.split("|", 1)]
        name = parts[0] if parts else "New Script"
        steps = []
        if len(parts) > 1:
            steps = [s.strip() for s in parts[1].split(";") if s.strip()]
        SCRIPTS.append((name, steps))
        return Action("notify", {"message": f"Added script: {name} ({len(steps)} steps)"})

    if cmd_lower.startswith("remove:"):
        payload = cmd.split(":", 1)[1].strip()
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(SCRIPTS):
                removed = SCRIPTS.pop(idx)
                # also remove from favorites if present
                favs = set(memory.get("favorites", []))
                if removed[0] in favs:
                    favs.discard(removed[0])
                    memory["favorites"] = list(favs)
                return Action("notify", {"message": f"Removed: {removed[0]}"})
            return Action("notify", {"message": "Index out of range."})
        for i, s in enumerate(SCRIPTS):
            if s[0].lower() == payload.lower():
                removed = SCRIPTS.pop(i)
                favs = set(memory.get("favorites", []))
                if removed[0] in favs:
                    favs.discard(removed[0])
                    memory["favorites"] = list(favs)
                return Action("notify", {"message": f"Removed: {removed[0]}"})
        return Action("notify", {"message": "Script not found."})

    if cmd_lower in {"list", "scripts"}:
        lines = [f"{i+1}. {s[0]} ({len(s[1])} steps)" for i, s in enumerate(SCRIPTS)]
        return Action("notify", {"message": "Scripts:\n" + "\n".join(lines)})

    if cmd_lower in {"best", "top", "best_script"}:
        stats = memory.get("stats", {})
        if not stats:
            return Action("notify", {"message": "No stats yet."})
        def rate(d):
            t = int(d.get("total", 0))
            return int(d.get("helped", 0)) / t if t > 0 else 0.0
        best = max(stats.items(), key=lambda kv: (rate(kv[1]), int(kv[1].get("total",0))))
        name, d = best
        return Action("notify", {"message": f"Best script: {name} -> helped={d.get('helped',0)} total={d.get('total',0)}"})

    if cmd_lower.startswith("export"):
        return Action("export_csv", {})

    if cmd_lower.startswith("fav:") or cmd_lower.startswith("favorite:"):
        payload = cmd.split(":", 1)[1].strip() if ':' in cmd else ''
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(SCRIPTS):
                name = SCRIPTS[idx][0]
                favs = set(memory.get("favorites", []))
                if name in favs:
                    favs.discard(name)
                    action_msg = f"Unfavorited: {name}"
                else:
                    favs.add(name)
                    action_msg = f"Favorited: {name}"
                memory["favorites"] = list(favs)
                return Action("notify", {"message": action_msg})
        return Action("notify", {"message": "Provide index to favorite (e.g., 'fav: 2')."})

    if cmd_lower in {"favorites", "favs"}:
        favs = memory.get("favorites", [])
        if not favs:
            return Action("notify", {"message": "No favorites set."})
        return Action("notify", {"message": "Favorites:\n" + "\n".join(favs)})

    if cmd_lower.startswith("search:"):
        q = cmd.split(":", 1)[1].strip().lower() if ':' in cmd else ''
        if not q:
            return Action("notify", {"message": "Usage: search: keyword"})
        matches = [f"{i+1}. {s[0]}" for i, s in enumerate(SCRIPTS) if q in s[0].lower() or any(q in step.lower() for step in s[1])]
        if not matches:
            return Action("notify", {"message": "No matches."})
        return Action("notify", {"message": "Matches:\n" + "\n".join(matches)})

    if cmd_lower.startswith("show:"):
        payload = cmd.split(":", 1)[1].strip() if ':' in cmd else ''
        if payload.isdigit():
            idx = int(payload) - 1
            if 0 <= idx < len(SCRIPTS):
                name, steps = SCRIPTS[idx]
                return Action("notify", {"message": _format_script(name, steps)})
            return Action("notify", {"message": "Index out of range."})
        return Action("notify", {"message": "Usage: show: INDEX"})

    # default: pick a script
    idx = pick_script(memory)
    name, steps = SCRIPTS[idx]
    memory["last_script"] = name
    msg = _format_script(name, steps)
    return Action("notify", {"message": msg})

def _countdown(secs: int):
    for i in range(secs, 0, -1):
        print(f"  Starting in {i}...", end='












































run_agent()        user_text = Tools.ask("Another block? (or stop / commands)")        env.execute(Action("save_memory", {}), memory)            memory["stats"] = stats            stats[name] = d                d["helped"] = int(d.get("helped", 0)) + 1            if ans.lower().startswith("y"):            d["total"] = int(d.get("total", 0)) + 1            d = stats.get(name, {"helped": 0, "total": 0})            name = memory.get("last_script", "unknown")            stats = memory.get("stats", {})            ans = Tools.ask("Did it help? (yes/no)")            _countdown(COUNTDOWN_SECS)            # Small pause or countdown to simulate 2 minutes (demo shortened)        if out["result"] == "notified":        # If a script was displayed, offer optional countdown timer before asking            Tools.notify("Exported stats to CSV.")        if out["result"] == "exported_csv":        # If exported            break            Tools.notify("Saved rescue stats. Bye!")            env.execute(Action("save_memory", {}), memory)        if out["result"] == "terminated":        memory = out["memory"]        out = env.execute(action, memory)        action = decide_next_action(user_text, memory)    while True:    user_text = Tools.ask("What's blocking you? (Enter to pick)")    Tools.notify("Describe what you're stuck on. Type 'stop' to end. Type 'list' to view scripts.")    Tools.notify("2-Minute Rescue Agent is running.")    memory = env.execute(Action("load_memory", {}), {})["memory"] or {}    env = Environment(MEMORY_FILE)def run_agent():    print(' ' * 40, end='
')        time.sleep(1)')

SyntaxError: unterminated string literal (detected at line 107) (3846831413.py, line 107)

## Easy extensions

- Add a countdown timer (set `DEMO_MODE=True` to shorten).
- Let students add/remove their own rescue scripts at runtime using `add:` and `remove:`.
- Show best success-rate script via `best` command.
- Export script success stats to CSV with `export`.
- List current scripts and indices using `list` or `scripts`.
- Mark favorites with `fav: INDEX` and list them with `favorites` (favorites are weighted higher when picking).
- Search scripts by keyword using `search: KEYWORD`.
- Show steps for a script using `show: INDEX`.

Usage examples:

- `add: Quick Fix | Try a small change; Run tests` -> adds a script with two steps.
- `remove: 3` -> removes the 3rd script.
- `fav: 2` -> toggle favorite for script #2.
- `favorites` -> show favorited scripts.
- `search: minimal` -> find scripts matching 'minimal'.
- `show: 1` -> display steps for script #1.
- `export` -> writes `02_rescue_memory.csv` with success rates.